In [ ]:
from joblib import dump, load
import torch.utils.data as Data
import numpy as np
import torch
import torch.nn as nn
torch.manual_seed(100)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name="MKTCN"
dataset = "./pipeline_datasets/"
def dataloader(batch_size, workers=2):
    train_xdata = load(dataset+"trainX")
    train_ylabel = load(dataset+"trainY")
    val_xdata = load(dataset+"valX")
    val_ylabel = load(dataset+"valY")
    test_xdata = load(dataset+"testX")
    test_ylabel = load(dataset+"testY")
    train_loader = Data.DataLoader(dataset=Data.TensorDataset(train_xdata, train_ylabel), batch_size=batch_size, shuffle=True, num_workers=workers, drop_last=True)
    val_loader = Data.DataLoader(dataset=Data.TensorDataset(val_xdata, val_ylabel), batch_size=batch_size, shuffle=True, num_workers=workers, drop_last=True)
    test_loader = Data.DataLoader(dataset=Data.TensorDataset(test_xdata, test_ylabel), batch_size=batch_size, shuffle=True, num_workers=workers, drop_last=True)
    return train_loader, val_loader, test_loader


batch_size = 64

train_loader, val_loader, test_loader = dataloader(batch_size)

In [ ]:
import torch
import math

class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,

    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0] 
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid) 

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):

        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1 
        ) 
        B = y.transpose(0, 1)  
        solution = torch.linalg.lstsq( 
            A, B
        ).solution 
        result = solution.permute(
            2, 0, 1
        ) 

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):

        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor): 

        assert x.dim() == 2 and x.size(1) == self.in_features

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        return base_output + spline_output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin=0.01): 
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  
        splines = splines.permute(1, 0, 2)  
        orig_coeff = self.scaled_spline_weight 
        orig_coeff = orig_coeff.permute(1, 2, 0) 
        unreduced_spline_output = torch.bmm(splines, orig_coeff) 
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        ) 

        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype=torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )

        self.grid.copy_(grid.T) 
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):

        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )

from torch.nn.utils import weight_norm

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()


from torch.nn.utils import parametrizations

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.5):
        super(TemporalBlock, self).__init__()

        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))

        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)


        self.conv2 =weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)


        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None

        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        out = out + res
        out = self.relu(out)
        return out

class TCNKANModel(nn.Module):
    def __init__(self,  input_dim, num_classes, num_channels, kernel_size, dropout= 0.5):
        super().__init__()

        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = input_dim if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size, padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        
        self.network = nn.Sequential(*layers)

        self.avgpool = nn.AdaptiveAvgPool1d(1)

        grid_range=[-1, 1]
        self.kan_layer =  KANLinear(
                    num_channels[-1],
                    num_classes,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )

    def forward(self,input_seq): 
        batch_size = input_seq.size(0)

        input_seq = input_seq.view(batch_size, 150, 1)


        tcn_features = self.network(input_seq) 

        output_avgpool = self.avgpool(tcn_features) 
        output_avgpool = output_avgpool.reshape(batch_size, -1) 
        output = self.kan_layer(output_avgpool)  
        return output
    
    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):

        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )


In [ ]:
model = TCNKANModel(input_dim, num_classes, num_channels, kernel_size, dropout)  


optimizer = torch.optim.Adam(model.parameters(), learn_rate) 

In [ ]:
import time
import matplotlib.pyplot as plt

def model_train(batch_size, epochs, model, optimizer, loss_function, train_loader, val_loader):
    model = model.to(device)
    train_size = len(train_loader) * batch_size
    val_size = len(val_loader) * batch_size


    best_accuracy = 0.0
    best_model = model

    train_loss = [] 
    train_acc = [] 
    validate_acc = []
    validate_loss = []

    start_time = time.time()
    for epoch in range(epochs):
        model.train()

        loss_epoch = 0.
        correct_epoch = 0
        for seq, labels in train_loader: 
            seq, labels = seq.to(device), labels.to(device)

            optimizer.zero_grad()

            y_pred = model(seq) 

            probabilities = F.softmax(y_pred, dim=1)

            predicted_labels = torch.argmax(probabilities, dim=1)

            correct_epoch += (predicted_labels == labels).sum().item()

            loss = loss_function(y_pred, labels)
            loss_epoch += loss.item()

            loss.backward()
            optimizer.step()

        train_Accuracy  = correct_epoch/train_size 
        train_loss.append(loss_epoch/train_size)
        train_acc.append(train_Accuracy)
        print(f'Epoch: {epoch+1:2} train_Loss: {loss_epoch/train_size:10.8f} train_Accuracy:{train_Accuracy:4.4f}')

        with torch.no_grad():

            model.eval()
            loss_validate = 0.
            correct_validate = 0
            for data, label in val_loader:
                data, label = data.to(device), label.to(device)
                pre = model(data)

                probabilities = F.softmax(pre, dim=1)

                predicted_labels = torch.argmax(probabilities, dim=1)

                correct_validate += (predicted_labels == label).sum().item()
                loss = loss_function(pre, label)
                loss_validate += loss.item()

            val_accuracy = correct_validate/val_size 
            print(f'Epoch: {epoch+1:2} val_Loss:{loss_validate/val_size:10.8f},  validate_Acc:{val_accuracy:4.4f}')
            validate_loss.append(loss_validate/val_size)
            validate_acc.append(val_accuracy)

            if val_accuracy > best_accuracy:
                best_accuracy = val_accuracy
                best_model = model


    torch.save(best_model.state_dict(), 'best_model_tcn_kan.pt')
   
    print(f'\nDuration: {time.time() - start_time:.0f} seconds')
    plt.plot(range(epochs), train_loss, color = 'b',label = 'train_loss')
    plt.plot(range(epochs), train_acc, color = 'g',label = 'train_acc')
    plt.plot(range(epochs), validate_loss, color = 'y',label = 'validate_loss')
    plt.plot(range(epochs), validate_acc, color = 'r',label = 'validate_acc')

    plt.legend()
    plt.savefig("./"+model_name+"/loss.eps", format='eps')
    plt.savefig("./"+model_name+"/loss.pdf", format='pdf')
    plt.show() 
    print("best_accuracy :", best_accuracy)
    np.savetxt("./"+model_name+"/train_loss.txt", train_loss)
    np.savetxt("./"+model_name+"/train_acc.txt", train_acc)
    np.savetxt("./"+model_name+"/validate_acc.txt", validate_acc)
    np.savetxt("./"+model_name+"/train_loss.txt", train_loss)
    np.savetxt("./"+model_name+"/validate_loss.txt", validate_loss)

epochs = 10

model_train(batch_size, epochs, model, optimizer, loss_function, train_loader, val_loader)

In [ ]:
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F
import torch
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

model.load_state_dict(torch.load('best_model_tcn_kan.pt'))
model = model.to(device)


class_labels = []
predicted_labels = []
predicted_probabilities = []

with torch.no_grad():
    for test_data, test_label in test_loader:

        model.eval()
        test_data = test_data.to(device)
        test_output = model(test_data)
        probabilities = F.softmax(test_output, dim=1)
        predicted = torch.argmax(probabilities, dim=1)
        class_labels.extend(test_label.tolist())
        predicted_labels.extend(predicted.tolist())
        predicted_probabilities.extend(probabilities.tolist())


confusion_mat = confusion_matrix(class_labels, predicted_labels)

from sklearn.metrics import classification_report

report = classification_report(class_labels, predicted_labels, digits=4)
print(report)

class_labels_2d = np.array(class_labels)[:, np.newaxis]
predicted_labels_2d = np.array(predicted_labels)[:, np.newaxis]
predicted_probabilities_2d = np.array(predicted_probabilities)
combined_matrix = np.hstack((class_labels_2d, predicted_labels_2d, predicted_probabilities_2d))
with open("./" + model_name + "/reports.txt", 'w') as f:
    f.write(report)
np.savetxt("./"+model_name+"/label&probabilities.txt", combined_matrix)